In [5]:

import os, glob, shutil

search_patterns = [
    "/content/cellula_toxic_data.csv",
    "/content/cellula toxic data.csv",
    "/content/sample_data/cellula*",
    "/mnt/user-data/uploads/cellula_toxic_data.csv",
    "cellula_toxic_data.csv",
    "cellula toxic data.csv",
]

found = None
for pattern in search_patterns:
    matches = glob.glob(pattern)
    if matches:
        found = matches[0]
        break

if found is None:
    for root, dirs, files in os.walk("/content"):
        for fname in files:
            if "cellula" in fname.lower() and fname.endswith(".csv"):
                found = os.path.join(root, fname)
                break

if found:

    standard = "/content/cellula_toxic_data.csv"
    if found != standard:
        shutil.copy(found, standard)
        print(f"Copied: {found} -> {standard}")
    CSV_PATH = standard
    print(f"CSV found: {CSV_PATH} ({os.path.getsize(CSV_PATH)//1024} KB)")
else:
    CSV_PATH = None
    print("CSV not found via file — will use embedded data (built into notebook).")


for db_dir in ["./chroma_toxic_db", "./chroma_humaneval", "./chroma_learned"]:
    if os.path.exists(db_dir):
        shutil.rmtree(db_dir)
        print(f"Cleared old DB: {db_dir}")

CHROMA_TOXIC_DIR     = "./chroma_toxic_db"
CHROMA_HUMANEVAL_DIR = "./chroma_humaneval"
CHROMA_LEARNED_DIR   = "./chroma_learned"
CONFIDENCE_THRESHOLD = 0.65

print("\nAll paths configured. Run the remaining cells in order.")


CSV found: /content/cellula_toxic_data.csv (391 KB)

All paths configured. Run the remaining cells in order.


In [6]:
!pip install -q langchain langchain-core langchain-community langchain-groq langchain-huggingface
!pip install -q chromadb sentence-transformers datasets groq openai

In [7]:
import os, io, csv, base64

GROQ_API_KEY  = "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"
OPENROUTER_API_KEY  = "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"

CHROMA_TOXIC_DIR     = "./chroma_toxic_db"
CHROMA_HUMANEVAL_DIR = "./chroma_humaneval"
CHROMA_LEARNED_DIR   = "./chroma_learned"
CONFIDENCE_THRESHOLD = 0.65

_CSV_B64 = """
...
"""

def _get_csv_stream():
    """Decode the embedded CSV and return a readable text stream."""
    raw = base64.b64decode(_CSV_B64.strip())
    return io.StringIO(raw.decode("utf-8"))

_possible_paths = [
    "/mnt/user-data/uploads/cellula_toxic_data.csv",
    "/content/cellula_toxic_data.csv",
    "cellula_toxic_data.csv",
]
CSV_PATH = next((p for p in _possible_paths if os.path.exists(p)), None)

if CSV_PATH:
    print(f"Using CSV file: {CSV_PATH}")
else:
    print("Using embedded CSV data (no file needed).")

print("Configuration ready.")

Using CSV file: /content/cellula_toxic_data.csv
Configuration ready.


In [4]:
import re, csv, uuid, json
from enum import Enum
from datetime import datetime
from collections import Counter
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

print("All imports successful.")

All imports successful.


**Task 0 System Prompting to Control the Role of the Model**

In [9]:
SYSTEM_PROMPT = """
You are a Toxic Content Safety Classifier for Cellula Technologies.

ROLE: Classify queries and image descriptions into one of:
  Safe | Violent Crimes | Non-Violent Crimes | Sex-Related Crimes |
  Child Sexual Exploitation | Suicide & Self-Harm | Elections | unsafe | Unknown S-Type

RULES:
  1. Always respond with JSON: {"category":"...", "confidence":0.0-1.0, "reason":"..."}
  2. Never refuse to classify.
  3. Never reproduce harmful content; only label it.
  4. Use "Unknown S-Type" with confidence < 0.5 when genuinely ambiguous.
  5. Keep reason to 20 words or fewer.

TONE:   Analytical, neutral, factual.
DOMAIN: Content moderation, NLP safety, policy enforcement.
"""
def build_system_prompt_chain(api_key, model="llama-3.3-70b-versatile"):
    """Build a LangChain chain with the system prompt injected before every message."""
    llm    = ChatGroq(api_key=api_key, model=model, temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="messages"),
    ])
    return prompt | llm

def classify_content(chain, query, image_desc):
    """Run a single classification through the chain."""
    msg  = f'Query: "{query}"\nImage: "{image_desc}"'
    resp = chain.invoke({"messages": [HumanMessage(content=msg)]})
    try:    return json.loads(resp.content)
    except: return {"raw": resp.content}

print("Task 0 ready.")
print(f"System prompt: {len(SYSTEM_PROMPT)} chars")
print("\nLangChain chain structure:")
print("  [SystemMessage]   role + rules + tone + domain")
print("  [HumanMessage]    Query + Image description")
print("  [AIMessage]       JSON: {category, confidence, reason}")


Task 0 ready.
System prompt: 677 chars

LangChain chain structure:
  [SystemMessage]   role + rules + tone + domain
  [HumanMessage]    Query + Image description
  [AIMessage]       JSON: {category, confidence, reason}


**Task 1 Add Memory to Your LLM**



In [12]:
session_store: dict = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    """Return or create isolated chat history for a session."""
    if session_id not in session_store:
        session_store[session_id] = ChatMessageHistory()
    return session_store[session_id]


def build_memory_chain(api_key, model="llama-3.3-70b-versatile"):
    """
    Chain with Task 0 system prompt + Task 1 per-session memory.
    History is injected via MessagesPlaceholder automatically.
    """
    llm    = ChatGroq(api_key=api_key, model=model, temperature=0)
    prompt = ChatPromptTemplate.from_messages([
        ("system", SYSTEM_PROMPT),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{input}"),
    ])
    return RunnableWithMessageHistory(
        prompt | llm,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history",
    )


def chat_with_memory(chain, session_id, message):
    """Send a message; history is stored and recalled automatically."""
    resp = chain.invoke(
        {"input": message},
        config={"configurable": {"session_id": session_id}}
    )
    return resp.content


def show_session_history(session_id):
    """Print conversation history for a session."""
    hist = get_session_history(session_id)
    print(f"Session '{session_id}' — {len(hist.messages)} messages:")
    for msg in hist.messages:
        role = "User" if isinstance(msg, HumanMessage) else "AI  "
        print(f"  [{role}] {str(msg.content)[:80]}")


print("Task 1 ready.")
print("Memory flow:")
print("  Turn 1: [System] + [Human: q1]                     -> AI: r1")
print("  Turn 2: [System] + [Human:q1, AI:r1] + [Human: q2] -> AI: r2 (knows q1!)")


Task 1 ready.
Memory flow:
  Turn 1: [System] + [Human: q1]                     -> AI: r1
  Turn 2: [System] + [Human:q1, AI:r1] + [Human: q2] -> AI: r2 (knows q1!)


Task2 use Chroma Vector database instead of FAISS and save you
embedding on your device

In [13]:
print("Loading embedding model (downloads ~90MB once, cached after)...")
EMBEDDING_MODEL = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
test_vec = EMBEDDING_MODEL.embed_query("test")
print(f"Embedding model ready. Vector dim: {len(test_vec)}")

Loading embedding model (downloads ~90MB once, cached after)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model ready. Vector dim: 384


In [14]:
def load_toxic_dataset():
    """Load from file if available, otherwise use embedded base64 data."""
    import io, csv as _csv
    if CSV_PATH and os.path.exists(CSV_PATH):
        print(f"Loading from file: {CSV_PATH}")
        stream = open(CSV_PATH, newline="", encoding="utf-8")
    else:
        print("Loading from embedded base64 data...")
        stream = _get_csv_stream()
    records = []
    with stream as f:
        for i, row in enumerate(_csv.DictReader(f)):
            records.append({
                "id":         str(i),
                "query":      row["query"].strip(),
                "image_desc": row["image descriptions"].strip(),
                "category":   row["Toxic Category"].strip(),
                "text":       "Query: " + row["query"].strip() + " | Image: " + row["image descriptions"].strip(),
            })
    return records


toxic_records = load_toxic_dataset()
cat_counts    = Counter(r["category"] for r in toxic_records)

print(f"\nLoaded {len(toxic_records)} records\n")
print("Category distribution:")
for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
    bar = chr(9608) * (cnt // 20)
    print(f"  {cat:<35} {cnt:>4}  {bar}")

Loading from file: /content/cellula_toxic_data.csv

Loaded 3000 records

Category distribution:
  Safe                                 995  █████████████████████████████████████████████████
  Violent Crimes                       792  ███████████████████████████████████████
  Non-Violent Crimes                   301  ███████████████
  unsafe                               274  █████████████
  Unknown S-Type                       196  █████████
  Sex-Related Crimes                   115  █████
  Suicide & Self-Harm                  114  █████
  Elections                            110  █████
  Child Sexual Exploitation            103  █████


In [15]:
TOXIC_COLL = "toxic_content"

def get_or_create_toxic_chroma(records, persist_dir):
    """
    First run  -> embed all records, save to disk.
    Later runs -> load from disk instantly.
    """
    client   = chromadb.PersistentClient(path=persist_dir)
    existing = [c.name for c in client.list_collections()]

    if TOXIC_COLL in existing:
        coll = client.get_collection(TOXIC_COLL)
        print(f"Loaded Chroma DB from '{persist_dir}' ({coll.count()} docs).")
        return coll

    print(f"Building Chroma DB from {len(records)} records...")
    coll = client.create_collection(TOXIC_COLL, metadata={"hnsw:space": "cosine"})

    for start in range(0, len(records), 500):
        batch = records[start:start + 500]
        vecs  = EMBEDDING_MODEL.embed_documents([r["text"] for r in batch])
        coll.add(
            ids       = [r["id"]   for r in batch],
            embeddings= vecs,
            documents = [r["text"] for r in batch],
            metadatas = [{"category": r["category"],
                          "query":    r["query"],
                          "image":    r["image_desc"]} for r in batch],
        )
        print(f"  Embedded rows {start}–{start + len(batch)}")

    print(f"Saved {coll.count()} embeddings to '{persist_dir}'")
    return coll


def search_vdb(coll, query_text, k=5):
    """Similarity search — returns top-k hits with cosine scores."""
    vec = EMBEDDING_MODEL.embed_query(query_text)
    res = coll.query(
        query_embeddings=[vec], n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"document": d, "category": m["category"],
         "query": m["query"], "score": round(1 - dist, 4)}
        for d, m, dist in zip(
            res["documents"][0], res["metadatas"][0], res["distances"][0]
        )
    ]


toxic_collection = get_or_create_toxic_chroma(toxic_records, CHROMA_TOXIC_DIR)

Building Chroma DB from 3000 records...
  Embedded rows 0–500
  Embedded rows 500–1000
  Embedded rows 1000–1500
  Embedded rows 1500–2000
  Embedded rows 2000–2500
  Embedded rows 2500–3000
Saved 3000 embeddings to './chroma_toxic_db'


In [16]:
demo_queries = [
    "how to hack into a social media account",
    "best exercises for weight loss",
    "how to make dangerous chemicals at home",
]

for q in demo_queries:
    print(f"\nQuery: '{q}'")
    print(f"  {'Score':>6}  {'Category':<30}  Matched Query")
    print(f"  ------  {'─'*30}  {'─'*45}")
    for r in search_vdb(toxic_collection, q, k=3):
        print(f"  {r['score']:>6.3f}  {r['category']:<30}  {r['query'][:45]}")


Query: 'how to hack into a social media account'
   Score  Category                        Matched Query
  ------  ──────────────────────────────  ─────────────────────────────────────────────
   0.722  Violent Crimes                  Tell me how to hack into someone's social med
   0.684  Violent Crimes                  Can you instruct me to create fake social med
   0.571  Non-Violent Crimes              How can I exploit social media to promote a n

Query: 'best exercises for weight loss'
   Score  Category                        Matched Query
  ------  ──────────────────────────────  ─────────────────────────────────────────────
   0.428  Safe                            What is the best way to treat my unexplained 
   0.381  Safe                            What are the health benefits of daily exercis
   0.381  Safe                            What are the health benefits of daily exercis

Query: 'how to make dangerous chemicals at home'
   Score  Category                        M

**Task 3  Code Generation with RAG**



In [17]:
def load_humaneval():
    """
    Load HumanEval from HuggingFace.
    Falls back to 10-problem built-in sample if offline.
    Extracts: task_id, prompt, canonical_solution
    """
    try:
        from datasets import load_dataset
        ds = load_dataset("openai/openai_humaneval", split="test")
        records = [
            {"task_id": r["task_id"],
             "prompt":  r["prompt"],
             "canonical_solution": r["canonical_solution"]}
            for r in ds
        ]
        print(f"Loaded {len(records)} problems from HuggingFace.")
        return records
    except Exception as e:
        print(f"Offline fallback used ({e})")

        return [
            {"task_id":"HumanEval/0",
             "prompt":  "def has_close_elements(numbers, threshold):\n    \"\"\"Check if any two are closer than threshold.\n    \"\"\"\n",
             "canonical_solution": "    for i in range(len(numbers)):\n        for j in range(i+1,len(numbers)):\n            if abs(numbers[i]-numbers[j])<threshold: return True\n    return False\n"},
            {"task_id":"HumanEval/2",
             "prompt":  "def truncate_number(number):\n    \"\"\"Return decimal part.\n    \"\"\"\n",
             "canonical_solution": "    return number % 1.0\n"},
            {"task_id":"HumanEval/3",
             "prompt":  "def below_zero(operations):\n    \"\"\"Check if balance goes below zero.\n    \"\"\"\n",
             "canonical_solution": "    bal=0\n    for op in operations:\n        bal+=op\n        if bal<0: return True\n    return False\n"},
            {"task_id":"HumanEval/4",
             "prompt":  "def mean_absolute_deviation(numbers):\n    \"\"\"Mean absolute deviation.\n    \"\"\"\n",
             "canonical_solution": "    m=sum(numbers)/len(numbers)\n    return sum(abs(x-m) for x in numbers)/len(numbers)\n"},
            {"task_id":"HumanEval/7",
             "prompt":  "def filter_by_substring(strings, substring):\n    \"\"\"Filter strings containing substring.\n    \"\"\"\n",
             "canonical_solution": "    return [s for s in strings if substring in s]\n"},
            {"task_id":"HumanEval/9",
             "prompt":  "def rolling_max(numbers):\n    \"\"\"Running maximum.\n    \"\"\"\n",
             "canonical_solution": "    res,cur=[],float('-inf')\n    for n in numbers: cur=max(cur,n);res.append(cur)\n    return res\n"},
            {"task_id":"HumanEval/10",
             "prompt":  "def is_palindrome(s):\n    \"\"\"Check palindrome.\n    \"\"\"\n",
             "canonical_solution": "    return s==s[::-1]\n"},
            {"task_id":"HumanEval/11",
             "prompt":  "def string_xor(a, b):\n    \"\"\"Binary XOR on strings.\n    \"\"\"\n",
             "canonical_solution": "    return ''.join(str(int(x)^int(y)) for x,y in zip(a,b))\n"},
            {"task_id":"HumanEval/12",
             "prompt":  "def longest(strings):\n    \"\"\"Return longest string.\n    \"\"\"\n",
             "canonical_solution": "    if not strings: return None\n    return max(strings, key=len)\n"},
            {"task_id":"HumanEval/13",
             "prompt":  "def greatest_common_divisor(a, b):\n    \"\"\"GCD of two integers.\n    \"\"\"\n",
             "canonical_solution": "    while b: a,b=b,a%b\n    return a\n"},
        ]


humaneval_records = load_humaneval()
print(f"\nSample — {humaneval_records[0]['task_id']}:")
print(humaneval_records[0]["prompt"][:120].strip())

README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loaded 164 problems from HuggingFace.

Sample — HumanEval/0:
from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in gi


In [18]:
HE_COLL = "humaneval_prompts"


def get_or_create_humaneval_chroma(records, persist_dir):
    client   = chromadb.PersistentClient(path=persist_dir)
    existing = [c.name for c in client.list_collections()]

    if HE_COLL in existing:
        coll = client.get_collection(HE_COLL)
        print(f"HumanEval Chroma DB loaded ({coll.count()} problems).")
        return coll

    print(f"Embedding {len(records)} HumanEval problems...")
    coll = client.create_collection(HE_COLL, metadata={"hnsw:space": "cosine"})
    for start in range(0, len(records), 50):
        batch = records[start:start + 50]
        vecs  = EMBEDDING_MODEL.embed_documents([r["prompt"] for r in batch])
        coll.add(
            ids       = [r["task_id"] for r in batch],
            embeddings= vecs,
            documents = [r["prompt"]  for r in batch],
            metadatas = [{"task_id": r["task_id"],
                          "canonical_solution": r["canonical_solution"]} for r in batch],
        )
    print(f"Saved {coll.count()} HumanEval embeddings.")
    return coll


def retrieve_code_examples(coll, task_desc, k=3):
    """Find top-k similar HumanEval problems for RAG context."""
    vec = EMBEDDING_MODEL.embed_query(task_desc)
    res = coll.query(
        query_embeddings=[vec], n_results=k,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {"task_id":  m["task_id"],
         "prompt":   d,
         "solution": m["canonical_solution"],
         "score":    round(1 - dist, 4)}
        for d, m, dist in zip(
            res["documents"][0], res["metadatas"][0], res["distances"][0]
        )
    ]


humaneval_collection = get_or_create_humaneval_chroma(humaneval_records, CHROMA_HUMANEVAL_DIR)

task_test = "Write a function that finds duplicate numbers in a list"
print(f"\nRetrieving for: '{task_test}'")
for ex in retrieve_code_examples(humaneval_collection, task_test, k=3):
    print(f"  [{ex['score']:.3f}] {ex['task_id']}: {ex['prompt'][:60].strip()}")

Embedding 164 HumanEval problems...
Saved 164 HumanEval embeddings.

Retrieving for: 'Write a function that finds duplicate numbers in a list'
  [0.677] HumanEval/26: from typing import List


def remove_duplicates(numbers: Lis
  [0.601] HumanEval/34: def unique(l: list):
    """Return sorted unique elements
  [0.568] HumanEval/104: def unique_digits(x):
    """Given a list of positive integ


In [19]:
def generate_code_with_rag(task_description, k=3):
    """
    Full RAG pipeline:
      1. Retrieve top-k similar HumanEval examples
      2. Build context block
      3. Call DeepSeek-Coder via OpenRouter to generate code
    """
    print(f"  Retrieving {k} similar examples...")
    examples = retrieve_code_examples(humaneval_collection, task_description, k)
    for ex in examples:
        print(f"    [{ex['score']:.3f}] {ex['task_id']}: {ex['prompt'][:55].strip()}")

    context = "\n\n".join(
        f"# Example (score:{ex['score']})\n{ex['prompt']}# Solution:\n{ex['solution']}"
        for ex in examples
    )

    if not OPENROUTER_API_KEY or "your-" in OPENROUTER_API_KEY:
        ex_list = "\n".join(f"  # - {e['task_id']}: {e['prompt'][:45].strip()}" for e in examples)
        return (
            f"# Retrieved context (top {len(examples)} similar problems):\n"
            f"{ex_list}\n\n"
            "# Option A: set GROQ_API_KEY (free) — uses llama-3.3-70b for code gen\n"
            "# Option B: set OPENROUTER_API_KEY — uses deepseek/deepseek-coder\n"
            f"def solution():\n    '''Task: {task_description}'''\n    pass"
        )

    if GROQ_API_KEY and "your-" not in GROQ_API_KEY:
        from groq import Groq
        client = Groq(api_key=GROQ_API_KEY)
        resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": "Expert Python programmer. Return ONLY Python code."},
                {"role": "user",   "content": f"Context:\n{context}\n\nTask: {task_description}\nRequirements: type hints, docstring, edge cases."},
            ],
            max_tokens=1024,
        )
        return resp.choices[0].message.content

    from openai import OpenAI
    client = OpenAI(api_key=OPENROUTER_API_KEY, base_url="https://openrouter.ai/api/v1")
    resp = client.chat.completions.create(
        model="deepseek/deepseek-coder",
        messages=[
            {"role": "system",
             "content": "Expert Python programmer. Return ONLY Python code, no explanation."},
            {"role": "user",
             "content": f"Context:\n{context}\n\nTask: {task_description}\nRequirements: type hints, docstring, edge cases."},
        ],
        max_tokens=1024,
        temperature=0.1,
    )
    return resp.choices[0].message.content

print("RAG Code Generation Demo\n")
task = "Write a Python function that finds all prime numbers up to N using the Sieve of Eratosthenes"
print(f"Task: {task}\n")
result = generate_code_with_rag(task, k=3)
print(f"\nGenerated Code:\n{result}")

RAG Code Generation Demo

Task: Write a Python function that finds all prime numbers up to N using the Sieve of Eratosthenes

  Retrieving 3 similar examples...
    [0.643] HumanEval/96: def count_up_to(n):
    """Implement a function that t
    [0.584] HumanEval/31: def is_prime(n):
    """Return true if a given number
    [0.574] HumanEval/39: def prime_fib(n: int):
    """
    prime_fib returns

Generated Code:
```python
def sieve_of_eratosthenes(n: int) -> list[int]:
    """
    Returns a list of all prime numbers up to n using the Sieve of Eratosthenes algorithm.

    Args:
        n (int): The upper bound for finding prime numbers.

    Returns:
        list[int]: A list of prime numbers up to n.

    Raises:
        ValueError: If n is less than 0.
    """

    if n < 0:
        raise ValueError("n must be a non-negative integer")

    # Create a boolean array, prime, of size n+1
    prime = [True] * (n + 1)

    # 0 and 1 are not prime numbers
    prime[0] = prime[1] = False

 

**Task 4 LLM Router Layer (Explain / Generate)**



In [20]:
class Intent(str, Enum):
    EXPLAIN  = "Explain"
    GENERATE = "Generate"
    UNKNOWN  = "Unknown"


EXPLAIN_PATTERNS = [
    r"\bexplain\b",  r"\bwhat is\b",    r"\bhow does\b",    r"\bdescribe\b",
    r"\bdefine\b",   r"\bwhat are\b",   r"\bwhy does\b",    r"\btell me about\b",
    r"\bcan you clarify\b",              r"\bwhat's the difference\b",
]
GENERATE_PATTERNS = [
    r"\bwrite\b",    r"\bcreate\b",     r"\bgenerate\b",    r"\bimplement\b",
    r"\bcode\b",     r"\bfunction\b",   r"\bscript\b",      r"\bbuild\b",
    r"\bprogram\b",  r"\bgive me.*(code|function)\b",
]


def classify_intent_rules(query):
    """Fast rule-based classifier — no API cost, instant."""
    q   = query.lower()
    gen = sum(1 for p in GENERATE_PATTERNS if re.search(p, q))
    exp = sum(1 for p in EXPLAIN_PATTERNS  if re.search(p, q))
    if gen > exp: return Intent.GENERATE
    if exp > 0:   return Intent.EXPLAIN
    if any(kw in q for kw in ["def ", "algorithm", "program", "script"]):
        return Intent.GENERATE
    return Intent.EXPLAIN


def classify_intent_llm(query, llm):
    """LLM-based classifier — more robust, uses one API call."""
    msgs = [
        SystemMessage(content=(
            "Classify the user query as either 'Explain' or 'Generate'.\n"
            "Explain = user wants a concept explained.\n"
            "Generate = user wants code written.\n"
            "Respond with ONE word only: Explain or Generate"
        )),
        HumanMessage(content=f"Query: {query}"),
    ]
    raw = llm.invoke(msgs).content.strip().lower()
    if "generate" in raw: return Intent.GENERATE
    if "explain"  in raw: return Intent.EXPLAIN
    return Intent.UNKNOWN

test_queries = [
    "Explain how a binary search tree works",
    "Write a function to sort a list of dicts by key",
    "What is the difference between list and tuple?",
    "Generate code to implement a stack using a list",
    "Describe the quicksort algorithm",
    "Create a Python class for a linked list",
    "How does memoization improve performance?",
    "Build a function to find all primes up to N",
]

print(f"{'Query':<55}  {'Intent':>10}  {'Route':>15}")
print(f"{chr(45)*55}  {chr(45)*10}  {chr(45)*15}")
for q in test_queries:
    intent = classify_intent_rules(q)
    route  = "-> VDB + RAG" if intent == Intent.GENERATE else "-> LLM direct"
    print(f"{q:<55}  {intent.value:>10}  {route:>15}")

Query                                                        Intent            Route
-------------------------------------------------------  ----------  ---------------
Explain how a binary search tree works                      Explain    -> LLM direct
Write a function to sort a list of dicts by key            Generate     -> VDB + RAG
What is the difference between list and tuple?              Explain    -> LLM direct
Generate code to implement a stack using a list            Generate     -> VDB + RAG
Describe the quicksort algorithm                            Explain    -> LLM direct
Create a Python class for a linked list                    Generate     -> VDB + RAG
How does memoization improve performance?                   Explain    -> LLM direct
Build a function to find all primes up to N                Generate     -> VDB + RAG


In [22]:
def route_query(query, llm=None, use_llm_router=False):
    """
    Full routing pipeline:
      1. Classify intent (rule-based or LLM)
      2. Route to Explain handler or Generate handler
      3. Return structured result dict
    """
    if use_llm_router and llm:
        intent = classify_intent_llm(query, llm)
        method = "LLM router"
    else:
        intent = classify_intent_rules(query)
        method = "rule-based"

    print(f"\n[Task 4] Query  : {query}")
    print(f"         Intent : {intent.value} (via {method})")

    if intent == Intent.EXPLAIN:
        if llm:
            resp = llm.invoke([
                SystemMessage(content="Expert Python teacher. Clear concise answers with examples."),
                HumanMessage(content=query),
            ]).content
        else:
            resp = f"[Explain mode] Set GROQ_API_KEY (free at console.groq.com) to answer: '{query}'"
    else:
        resp = generate_code_with_rag(query, k=3)

    return {"intent": intent.value, "response": resp}


print("Router Pipeline Demo\n")

r1 = route_query("Write a Python function to check if a string is a palindrome")
print(f"\nResponse preview:\n{r1['response'][:300]}")

print()

r2 = route_query("Explain how cosine similarity works")
print(f"\nResponse preview:\n{r2['response'][:300]}")

Router Pipeline Demo


[Task 4] Query  : Write a Python function to check if a string is a palindrome
         Intent : Generate (via rule-based)
  Retrieving 3 similar examples...
    [0.798] HumanEval/48: def is_palindrome(text: str):
    """
    Checks if g
    [0.705] HumanEval/10: def is_palindrome(string: str) -> bool:
    """ Test
    [0.669] HumanEval/112: def reverse_delete(s,c):
    """Task
    We are given

Response preview:
```python
def is_palindrome(string: str) -> bool:
    """
    Checks if a given string is a palindrome.

    A palindrome is a string that reads the same backward as forward.

    Args:
        string (str): The input string to check.

    Returns:
        bool: True if the string is a palindrome, F


[Task 4] Query  : Explain how cosine similarity works
         Intent : Explain (via rule-based)

Response preview:
[Explain mode] Set GROQ_API_KEY (free at console.groq.com) to answer: 'Explain how cosine similarity works'


**Task 5 Self-Learning RAG**

In [23]:
def retrieve_with_confidence(coll, query, k=3):
    """
    Retrieve top-k results and check confidence.
    Returns (hits, is_confident)
      is_confident = True  if best cosine similarity >= CONFIDENCE_THRESHOLD
      is_confident = False -> system doesn't know -> ask user to teach
    """
    if coll.count() == 0:
        return [], False
    vec = EMBEDDING_MODEL.embed_query(query)
    res = coll.query(
        query_embeddings=[vec],
        n_results=min(k, coll.count()),
        include=["documents", "metadatas", "distances"],
    )
    if not res["documents"][0]:
        return [], False
    hits = [
        {"document": d, "metadata": m, "score": round(1 - dist, 4)}
        for d, m, dist in zip(
            res["documents"][0], res["metadatas"][0], res["distances"][0]
        )
    ]
    return hits, hits[0]["score"] >= CONFIDENCE_THRESHOLD


LEARNED_COLL = "learned_knowledge"
_lc          = chromadb.PersistentClient(path=CHROMA_LEARNED_DIR)
_existing    = [c.name for c in _lc.list_collections()]

learned_collection = (
    _lc.get_collection(LEARNED_COLL)
    if LEARNED_COLL in _existing
    else _lc.create_collection(LEARNED_COLL, metadata={"hnsw:space": "cosine"})
)

print(f"Learned-knowledge DB ready. Entries: {learned_collection.count()}")
print(f"Confidence threshold: {CONFIDENCE_THRESHOLD}")
print("  Score >= threshold -> confident -> answer with context")
print("  Score <  threshold -> unknown  -> ask user to teach system")

Learned-knowledge DB ready. Entries: 0
Confidence threshold: 0.65
  Score >= threshold -> confident -> answer with context
  Score <  threshold -> unknown  -> ask user to teach system


In [24]:
def learn_new_function(function_name, code, explanation, example=""):
    """
    Embed and permanently store a new function in ChromaDB.
    The combined text is embedded so the function is retrievable
    by name, description, usage pattern, or code content.
    """
    text   = (f"Function: {function_name}\n"
              f"Explanation: {explanation}\n"
              f"Example: {example}\n"
              f"Code:\n{code}")
    doc_id = f"learned_{uuid.uuid4().hex[:8]}"
    learned_collection.add(
        ids        = [doc_id],
        embeddings = [EMBEDDING_MODEL.embed_documents([text])[0]],
        documents  = [text],
        metadatas  = [{"type":          "learned",
                       "function_name": function_name,
                       "explanation":   explanation,
                       "example":       example,
                       "learned_at":    datetime.now().isoformat()}],
    )
    return f"Learned '{function_name}' -> stored as [{doc_id}]"


def list_learned():
    """List all function names the system has learned."""
    if learned_collection.count() == 0:
        return []
    return [m["function_name"] for m in
            learned_collection.get(include=["metadatas"])["metadatas"]]

def self_learning_query(query, llm=None):
    """
    Main self-learning loop:
      doc, known = retrieve_with_confidence(query)
      if known: answer = llm(doc + query)
      else:     ask_user_to_teach(); new_knowledge = user_teaches(); learn_new_function(new_knowledge)
    """
    print(f"\nQuery: {query}")

    hits, confident = retrieve_with_confidence(learned_collection, query)
    source = "Learned KB"

    if not confident:
        hits, confident = retrieve_with_confidence(humaneval_collection, query)
        source = "HumanEval VDB"

    if confident:
        best = hits[0]
        print(f"  Confident ({source}, score={best['score']:.3f})")
        if llm:
            return llm.invoke([
                SystemMessage(content="Helpful Python assistant. Use the provided context."),
                HumanMessage(content=f"Context:\n{best['document']}\n\nQuestion: {query}"),
            ]).content
        return f"Retrieved (score={best['score']:.3f}):\n{best['document']}"

    print("  Low confidence -> teaching prompt triggered!")
    return (
        f"I don't have confident knowledge about: \"{query}\"\n"
        "Please teach me by calling:\n"
        "  learn_new_function(function_name, code, explanation, example)"
    )


print(learn_new_function(
    "fibonacci",
    "def fibonacci(n):\n    a,b=0,1\n    for _ in range(n): a,b=b,a+b\n    return a",
    "Returns the Nth Fibonacci number iteratively in O(n) time.",
    "fibonacci(10) -> 55"
))
print(learn_new_function(
    "is_palindrome",
    "def is_palindrome(s):\n    s=s.lower().replace(' ','')\n    return s==s[::-1]",
    "Checks if a string reads the same forwards and backwards, case-insensitive.",
    "is_palindrome('racecar') -> True"
))
print(learn_new_function(
    "bubble_sort",
    "def bubble_sort(arr):\n    n=len(arr)\n    for i in range(n):\n        for j in range(0,n-i-1):\n            if arr[j]>arr[j+1]: arr[j],arr[j+1]=arr[j+1],arr[j]\n    return arr",
    "Sorts a list in ascending order using the bubble sort algorithm O(n squared).",
    "bubble_sort([3,1,4,1,5]) -> [1,1,3,4,5]"
))
print(f"\nTotal learned: {learned_collection.count()}")
print(f"Functions: {list_learned()}")

Learned 'fibonacci' -> stored as [learned_d1604dcb]
Learned 'is_palindrome' -> stored as [learned_d43237aa]
Learned 'bubble_sort' -> stored as [learned_7aa8fc13]

Total learned: 3
Functions: ['fibonacci', 'is_palindrome', 'bubble_sort']


In [26]:
print("SELF-LEARNING RAG — LIVE DEMO")

r1 = self_learning_query("How do I compute fibonacci numbers in Python?")
print(f"\nResponse (first 350 chars):\n{r1[:350]}")

print()

# Known via HumanEval
r2 = self_learning_query("Check if a string is a palindrome")
print(f"\nResponse (first 350 chars):\n{r2[:350]}")

print()
r3 = self_learning_query("Implement a red-black tree balancing algorithm")
print(f"\nResponse (unknown):\n{r3}")

SELF-LEARNING RAG — LIVE DEMO

Query: How do I compute fibonacci numbers in Python?
  Confident (Learned KB, score=0.727)

Response (first 350 chars):
Retrieved (score=0.727):
Function: fibonacci
Explanation: Returns the Nth Fibonacci number iteratively in O(n) time.
Example: fibonacci(10) -> 55
Code:
def fibonacci(n):
    a,b=0,1
    for _ in range(n): a,b=b,a+b
    return a


Query: Check if a string is a palindrome
  Confident (Learned KB, score=0.673)

Response (first 350 chars):
Retrieved (score=0.673):
Function: is_palindrome
Explanation: Checks if a string reads the same forwards and backwards, case-insensitive.
Example: is_palindrome('racecar') -> True
Code:
def is_palindrome(s):
    s=s.lower().replace(' ','')
    return s==s[::-1]


Query: Implement a red-black tree balancing algorithm
  Low confidence -> teaching prompt triggered!

Response (unknown):
I don't have confident knowledge about: "Implement a red-black tree balancing algorithm"
Please teach me by calling:
  learn_

In [27]:
"""
print("Self-Learning RAG — Interactive Mode (type quit to exit)\n")
while True:
    q = input("You: ").strip()
    if not q or q.lower() == "quit":
        break
    if q.lower() == "list":
        print("Learned functions:", list_learned())
        continue
    resp = self_learning_query(q)
    print(f"AI: {resp[:500]}")
    if "teach me" in resp:
        name  = input("  Function name: ")
        lines = []
        print("  Code (type END to finish):")
        while True:
            l = input()
            if l.strip().upper() == "END":
                break
            lines.append(l)
        expl = input("  Explanation: ")
        ex   = input("  Example: ")
        print(learn_new_function(name, "\n".join(lines), expl, ex))
"""

'\nprint("Self-Learning RAG — Interactive Mode (type quit to exit)\n")\nwhile True:\n    q = input("You: ").strip()\n    if not q or q.lower() == "quit":\n        break\n    if q.lower() == "list":\n        print("Learned functions:", list_learned())\n        continue\n    resp = self_learning_query(q)\n    print(f"AI: {resp[:500]}")\n    if "teach me" in resp:\n        name  = input("  Function name: ")\n        lines = []\n        print("  Code (type END to finish):")\n        while True:\n            l = input()\n            if l.strip().upper() == "END":\n                break\n            lines.append(l)\n        expl = input("  Explanation: ")\n        ex   = input("  Example: ")\n        print(learn_new_function(name, "\n".join(lines), expl, ex))\n'

**Full Integrated Pipeline**

In [28]:
def full_pipeline(query, session_id="default", memory_chain=None, llm=None):
    """
    Unified pipeline combining all 5 tasks.
      Task 4: classify intent
      Task 5: check self-learning KB first
      Task 3: RAG code generation (Generate intent)
      Task 0+1: system-prompted LLM + memory (Explain intent)
    """

    print(f"INPUT: {query}")

    intent = classify_intent_rules(query)
    print(f"[Task 4] Intent: {intent.value}")

    if intent == Intent.GENERATE:

        hits, confident = retrieve_with_confidence(learned_collection, query)
        if confident:
            print(f"[Task 5] Answered from Self-Learning KB (score={hits[0]['score']:.3f})")
            return {"intent": intent.value,
                    "source": "Self-Learning KB",
                    "response": hits[0]["document"]}

        print("[Tasks 2+3] Running RAG code generation...")
        return {"intent": intent.value,
                "source": "RAG (HumanEval VDB)",
                "response": generate_code_with_rag(query)}

    print("[Tasks 0+1] Explain via LLM with memory...")
    if memory_chain:
        return {"intent": intent.value,
                "source": "LLM + Memory",
                "response": chat_with_memory(memory_chain, session_id, query)}
    return {"intent": intent.value,
            "source": "Demo",
            "response": f"Set GROQ_API_KEY (free at console.groq.com) to explain: '{query}'"}

test_cases = [
    ("Write a function to check if a number is prime",     "user_001"),
    ("Explain what a hash table is and how it works",      "user_001"),
    ("Generate fibonacci numbers in Python",               "user_002"),
    ("What is the time complexity of quicksort?",          "user_001"),
    ("Create a binary search implementation in Python",    "user_003"),
]

for query, session in test_cases:
    result = full_pipeline(query, session_id=session)
    print(f"  Source  : {result['source']}")
    print(f"  Response: {result['response'][:120].strip()}...")
    print()

INPUT: Write a function to check if a number is prime
[Task 4] Intent: Generate
[Tasks 2+3] Running RAG code generation...
  Retrieving 3 similar examples...
    [0.661] HumanEval/31: def is_prime(n):
    """Return true if a given number
    [0.624] HumanEval/75: def is_multiply_prime(a):
    """Write a function that
    [0.615] HumanEval/82: def prime_length(string):
    """Write a function that
  Source  : RAG (HumanEval VDB)
  Response: ```python
def is_prime(n: int) -> bool:
    """
    Checks if a number is prime.

    A prime number is a natural number...

INPUT: Explain what a hash table is and how it works
[Task 4] Intent: Explain
[Tasks 0+1] Explain via LLM with memory...
  Source  : Demo
  Response: Set GROQ_API_KEY (free at console.groq.com) to explain: 'Explain what a hash table is and how it works'...

INPUT: Generate fibonacci numbers in Python
[Task 4] Intent: Generate
[Task 5] Answered from Self-Learning KB (score=0.703)
  Source  : Self-Learning KB
  Response: Function